<a href="https://colab.research.google.com/github/hayatosc/example-search/blob/main/search_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install lindera_py

In [7]:
import duckdb
import torch
from transformers import AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer
from lindera_py import Segmenter, Tokenizer, load_dictionary
import torch.nn.functional as F
import pandas as pd

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

v_model = SentenceTransformer("cl-nagoya/ruri-v3-70m", device=device)

dictionary = load_dictionary("ipadic")
segmenter = Segmenter("normal", dictionary)
tokenizer = Tokenizer(segmenter)

def fts_tokenizer(text):
    return " ".join(t.text for t in tokenizer.tokenize(text))

In [9]:
conn = duckdb.connect()
conn.sql('INSTALL VSS;')
conn.sql('LOAD VSS;')
conn.sql('INSTALL FTS;')
conn.sql('LOAD FTS;')
conn.sql("CREATE TABLE circles AS SELECT * FROM 'https://raw.githubusercontent.com/hayatosc/example-search/refs/heads/main/circleListFiction.csv';")
conn.sql('ALTER TABLE circles ADD COLUMN vss FLOAT[384];')
conn.sql('ALTER TABLE circles ADD COLUMN fts VARCHAR')
conn.sql('SHOW ALL TABLES;')



┌──────────┬─────────┬─────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database │ schema  │  name   │                                                                      column_names                                                                      │                                                               column_types                                                               │ temporary │
│ varchar  │ varchar │ varchar │                                                                       varchar[]                                                                        │                                                                varchar[]                                                                 │  

In [10]:
df = conn.sql('SELECT * FROM circles;').df()

for index, row in df.iterrows():
    row = row.fillna('')
    text = '団体名: ' + row['団体名'] + '\n\n主体としているキャンパス: ' + row['主体としているキャンパス'] + '\n\n団体の区分: ' + row['団体の区分'] + '\n\n活動内容: ' + row['活動内容'] + '\n\nひとこと:' + row['ひとこと'] + '\n\n活動場所: ' + row['活動場所'] + '\n\n年間日程: ' + row['年間日程']
    embedding = v_model.encode(text)
    conn.execute('UPDATE circles SET vss = ? WHERE 団体名 = ?', [embedding, row['団体名']])

    fts_text = fts_tokenizer(text)
    conn.execute('UPDATE circles SET fts = ? WHERE 団体名 = ?', [fts_text, row['団体名']])

conn.sql("""
    PRAGMA create_fts_index(
        'circles',
        '企画ID',
        'fts',

        stemmer = 'none',
        stopwords = 'none',
        ignore = '',
        lower = false,
        strip_accents = false
    );
    """)

conn.sql("SELECT * FROM circles;").show()

┌────────┬────────┬──────────────────────────────────────────────┬──────────────────────────┬────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────┬────────────────────────────────────┬─────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────

In [11]:
v_query_prefix = '検索クエリ: '

def vector_search(query):
    query = v_query_prefix + query
    with torch.inference_mode():
      query_embedding = v_model.encode(query)
      result = conn.sql(f"""
          SELECT 企画ID, array_cosine_distance(vss, {query_embedding.tolist()}::FLOAT[384]) as distance
          FROM circles
          ORDER BY distance ASC
      """).fetchall()
    return result

In [12]:
def fts_search(query):
    query_token = fts_tokenizer(query)
    rows = conn.sql(f"""
      SELECT 企画ID, fts_main_circles.match_bm25(企画ID, '{query_token}') as score
      FROM circles
      WHERE score IS NOT NULL
      ORDER BY score DESC
    """).fetchall()

    return rows

In [13]:
def search(query):
  print("--- DuckDB-FTS + Lindera ---")
  fts_rows = fts_search(query)
  for id, score in fts_rows[:10]:
      print(f"ID: {id}, Score: {score:.4f}")
  fts_ranks = {id: score for id, score in fts_rows}

  print("--- DuckDB-VSS + Ruri v3 ---")
  vss_rows = vector_search(query)
  for id, score in vss_rows[:10]:
      print(f"ID: {id}, Score: {score:.4f}")
  vss_ranks = {id: 1.0 / (score + 1e-6)  for id, score in vss_rows}

  print('\n\n\n\n')

  # Reciprocal Rank Fusion (FTSとVSSでスコアの意味が異なるため、調整)
  combined_ranks = {}
  for id in set(fts_ranks) | set(vss_ranks):
    fts_score = fts_ranks.get(id, 0)
    vss_score = vss_ranks.get(id, 0)

    # FTSとVSSのスコアを組み合わせて最終スコアを計算 (例: 重み付け平均)
    combined_ranks[id] = 0.5 * fts_score + 0.5 * vss_score

  sorted_results = sorted(combined_ranks.items(), key=lambda item: item[1], reverse=True)

  print('--- Hybrid Search ---')
  for id, score in sorted_results[:10]:
      print(f"ID: {id}, Score: {score:.4f}")

      original_data = conn.execute(f"SELECT 回答ID, 企画ID, 団体名, 主体としているキャンパス, 団体の区分, 活動内容, 活動日, 活動場所, 部員数, 部費, ひとこと, キャッチコピー, 年間日程 FROM circles WHERE 企画ID = '{id}'").fetchone()
      column_names = [desc[0] for desc in conn.execute(f"SELECT 回答ID, 企画ID, 団体名, 主体としているキャンパス, 団体の区分, 活動内容, 活動日, 活動場所, 部員数, 部費, ひとこと, キャッチコピー, 年間日程 FROM circles WHERE 企画ID = '{id}'").description]

      for column_name, value in zip(column_names, original_data):
          print(f"{column_name}: {value}")
      print("---")  # データ間の区切り線

In [15]:
search('音楽')

--- DuckDB-FTS + Lindera ---
ID: 2051, Score: 2.4337
ID: 2041, Score: 2.4169
ID: 2036, Score: 2.2723
ID: 2008, Score: 2.2382
ID: 2024, Score: 2.1981
ID: 2015, Score: 2.0660
ID: 2132, Score: 2.0293
ID: 2098, Score: 1.9176
ID: 2012, Score: 1.5039
ID: 2107, Score: 1.4391
--- DuckDB-VSS + Ruri v3 ---
ID: 2024, Score: 0.2153
ID: 2015, Score: 0.2155
ID: 2008, Score: 0.2227
ID: 2036, Score: 0.2283
ID: 2041, Score: 0.2303
ID: 2132, Score: 0.2311
ID: 2051, Score: 0.2341
ID: 2039, Score: 0.2376
ID: 2134, Score: 0.2376
ID: 2012, Score: 0.2381





--- Hybrid Search ---
ID: 2024, Score: 3.4215
回答ID: 1024
企画ID: 2024
団体名: 軽音楽サークル「Sound Scape」
主体としているキャンパス: 野田
団体の区分: 音楽系
活動内容: バンド結成、コピー・オリジナル曲練習。月1程度のライブハウスでのライブ、学内イベント出演。
活動日: バンド毎に週1-2回練習 + 月1ライブ(週末)
活動場所: 柏・松戸周辺の音楽スタジオ、ライブハウス、講義棟
部員数: 90名（男子60名、女子30名）
部費: 入部費3000円 + 半期2000円 (スタジオ代補助等)
ひとこと: 初心者・経験者不問！楽器やりたい人、歌いたい人、バンド組みたい人、大歓迎！ギター、ベース、ドラム、キーボード、ボーカル募集中！機材なくても相談乗ります！
キャッチコピー: 音で繋がる、最高の瞬間を！
年間日程: 4月:新歓ライブ＆楽器体験会 5月:バンド結成会、楽器購入相談会 6月:新人ライブ 7月:夏ライブ 8月:夏